### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [2]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [4]:
### Read all PDFs in the directory

def process_pdfs(pdf_directory:str):
    """Generate all pdfs in a directory

    Args:
        pdf_directory (str): directory of the folder containing the pdfs
    """
    all_documents = []
    dir = Path(pdf_directory)

    pdf_files = list(dir.glob("**/*.pdf"))
    print(f'Found {len(pdf_files)} pdfs to process!')
    for pdf_file in pdf_files:
        try:
            print(f"Processing {pdf_file.name}")
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages!")
        except Exception as e:
            print(f"PDFs could not be loaded! Error: {e}")

    print(f"\nTotal PDFs loaded {len(all_documents)}")
    return all_documents

all_pdf_documents = process_pdfs("../data")

Found 1 pdfs to process!
Processing iso27001.pdf
Loaded 26 pages!

Total PDFs loaded 26


In [5]:
all_pdf_documents[0]

Document(metadata={'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'source': '../data/pdfs/iso27001.pdf', 'file_path': '../data/pdfs/iso27001.pdf', 'total_pages': 26, 'format': 'PDF 1.4', 'title': 'ISO/IEC 27001:2022', 'author': 'ISO', 'subject': '', 'keywords': '', 'moddate': '2022-10-26T19:22:40+08:00', 'trapped': '', 'modDate': "D:20221026192240+08'00'", 'creationDate': "D:20220929175400+02'00'", 'page': 0, 'source_file': 'iso27001.pdf', 'file_type': 'pdf'}, page_content="Information security, cybersecurity \nand privacy protection\xa0— Information \nsecurity management systems — \nRequirements\nSécurité de l'information, cybersécurité et protection de la vie \nprivée\xa0— Systèmes de management de la sécurité de l'information — \nExigences\nINTERNATIONAL \nSTANDARD\nISO/IEC \n27001\nThird edition \n2022-10\nReference number \nISO/IEC 27001:2022(E)\n© ISO/IEC 2022\n--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`-

In [10]:
### Text Splitting get into chunks

def split_documents(documents, chunk_size = 1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators= ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f'Split {len(documents)} documnents into {len(split_docs)} chunks')
    if split_docs:
        print(f"\nExample Chunk:")
        print(f'Content: {split_docs[0].page_content[:200]}...')
        print(f'Metadata: {split_docs[0].metadata}')
    
    return split_docs

In [11]:
chunks = split_documents(all_pdf_documents)
chunks

Split 26 documnents into 77 chunks

Example Chunk:
Content: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèm...
Metadata: {'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'source': '../data/pdfs/iso27001.pdf', 'file_path': '../data/pdfs/iso27001.pdf', 'total_pages': 26, 'format': 'PDF 1.4', 'title': 'ISO/IEC 27001:2022', 'author': 'ISO', 'subject': '', 'keywords': '', 'moddate': '2022-10-26T19:22:40+08:00', 'trapped': '', 'modDate': "D:20221026192240+08'00'", 'creationDate': "D:20220929175400+02'00'", 'page': 0, 'source_file': 'iso27001.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'source': '../data/pdfs/iso27001.pdf', 'file_path': '../data/pdfs/iso27001.pdf', 'total_pages': 26, 'format': 'PDF 1.4', 'title': 'ISO/IEC 27001:2022', 'author': 'ISO', 'subject': '', 'keywords': '', 'moddate': '2022-10-26T19:22:40+08:00', 'trapped': '', 'modDate': "D:20221026192240+08'00'", 'creationDate': "D:20220929175400+02'00'", 'page': 0, 'source_file': 'iso27001.pdf', 'file_type': 'pdf'}, page_content="Information security, cybersecurity \nand privacy protection\xa0— Information \nsecurity management systems — \nRequirements\nSécurité de l'information, cybersécurité et protection de la vie \nprivée\xa0— Systèmes de management de la sécurité de l'information — \nExigences\nINTERNATIONAL \nSTANDARD\nISO/IEC \n27001\nThird edition \n2022-10\nReference number \nISO/IEC 27001:2022(E)\n© ISO/IEC 2022\n--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`

In [15]:
### embedding and vector store DB

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [17]:
class EmbeddingManager:
    """Handles document embedding generation using SenteenceTransformer
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager

        Args:
            model_name (str, optional): HuggingFace model name for sentence embeddings. Defaults to "all-MiniLM-L6-v2".
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model
        """
        try:
            print(f"Loading embedding model: {self.model}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts:List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts

        Args:
            texts (List[str]): List of text strings to embed

        Returns:
            numpy array of embeddings with shapr(len(text), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

### Initialize the embedding manager
 
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: None


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13251.13it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store
    """

    def __int__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store

        Args:
            collection_name (str, optional): Name of the chromaDB collection. Defaults to "all_pdf_documents".
            persist_directory (str, optional): Directory to persist the vector store. Defaults to "../data/vector_store".
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection
        """
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create a collection
            self.collection = self.client.get_or_create_collection(self.collection_name, metadata={"description": "PDF doc embeddings for RAG"})
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in the Collection: {(self.collection.count())}")

        except Exception as e:
            print(f"Error initialiing collection Error: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and the embeddings to the vector store.

        Args:
            documents (List[Any]): List of languaging documents
            embeddings (np.ndarray): Corresponding embeddings for the document.
        """ 
        if len(embeddings) != len(embeddings):
            raise ValueError("Docs and embedding numbers dont match")
        print(f"loading {len(documents)} to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate uuid
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
    
